# 🎙️ EGY-Emilia · Pipeline notebook

Run the pipeline **one stage per cell**, so you can watch each stage's progress on its own.

1. **Part A: sample test.** Runs every stage on **one** sample video in a separate `sample/` folder,
   checks the results automatically, and lets you **listen** to kept and rejected chunks.
2. **Part B: full run.** Runs on everything in `sources.txt`, then publishes the dataset to HuggingFace as **private**.
   Part B refuses to start until Part A has passed.

Every stage is resumable: if a cell is interrupted, run it again and it continues where it stopped.

> **Kernel:** this notebook must run in the `egy` conda env, the one with NeMo and torch installed.
> One-time setup in the RunPod terminal:
> ```bash
> conda activate egy
> pip install ipykernel ipywidgets
> python -m ipykernel install --user --name egy --display-name "Python (egy)"
> ```
> Then pick **Kernel → Change kernel → Python (egy)** (reload the page if it is not listed).

## 0 · Setup

In [ ]:
import copy, gc, json, os, random, shutil, subprocess, sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / "config.yaml").exists(), "Open this notebook from the Egy-Emilia repo root."
sys.path.insert(0, str(REPO))

from IPython.display import Audio, display
from egy_emilia.config import load_config, resolve
from egy_emilia import download, diarize_stage, loudness, quality, transcribe, publish
import yt_dlp

cfg = load_config()

print("python :", sys.executable)
print("yt-dlp :", yt_dlp.version.__version__)
print("ffmpeg :", shutil.which("ffmpeg"))
try:
    import torch
    print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    torch = None
    print("torch  : NOT INSTALLED (wrong kernel?)")
if "egy" not in sys.executable:
    print("\n⚠️  This kernel does not look like the `egy` env; see the kernel note at the top.")
print("\nYouTube mode: js_runtime =", cfg.download.js_runtime, "| pot_provider =", cfg.download.pot_provider,
      "(both false = plain yt-dlp, which works on RunPod)")

In [ ]:
# ---- helpers used by the cells below ----
def free_gpu():
    """Release models from the previous stage before loading the next one."""
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

def probe(path):
    """(sample_rate, channels, duration_s) of an audio file, via ffprobe."""
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "stream=sample_rate,channels:format=duration",
         "-of", "json", str(path)], capture_output=True, text=True).stdout
    d = json.loads(out)
    s = d["streams"][0]
    return int(s["sample_rate"]), int(s["channels"]), float(d["format"]["duration"])

def run_download(c):
    """download.run, but a YouTube bot-block becomes a readable error instead of SystemExit."""
    try:
        download.run(c)
    except SystemExit:
        raise RuntimeError("YouTube blocked the download; see the messages above.") from None

def load_records(c):
    out = resolve(c.paths.output)
    all_ = json.loads((out / "chunks.json").read_text(encoding="utf-8"))
    clean = json.loads((out / "chunks_clean.json").read_text(encoding="utf-8"))
    return out, all_, clean

def metric(c, r):
    m = c.quality.filter_metric
    return r.get("overall") if m == "overall" else r["scores"].get(m)

def listen(c, records, n, title, sort_low=False):
    out = resolve(c.paths.output)
    pick = sorted(records, key=lambda r: metric(c, r) or 0)[:n] if sort_low else random.sample(records, min(n, len(records)))
    print(f"\n{title}  ({len(pick)} of {len(records)})")
    for r in pick:
        m = metric(c, r)
        print(f"• {r['chunk_path']}  |  {r['duration']:.1f}s  |  {c.quality.filter_metric}="
              f"{m if m is None else round(m, 2)}")
        if r.get("text"):
            print("  📝", r["text"])
        display(Audio(filename=str(out / r["chunk_path"])))

def hours(records):
    return sum(r["duration"] for r in records) / 3600

---
# Part A · Sample test (one video)

Everything in Part A is written to `sample/`, separate from the real `input_audios/`, `output/` and `.state/`.
Change `SAMPLE_URL` to test another video. `FRESH = True` wipes `sample/` first, for a clean retest.

In [ ]:
SAMPLE_URL = "https://youtu.be/zqqYdaWp13Q"
FRESH = True

SAMPLE = REPO / "sample"
if FRESH and SAMPLE.exists():
    shutil.rmtree(SAMPLE)
SAMPLE.mkdir(exist_ok=True)
(SAMPLE / "sources.txt").write_text(SAMPLE_URL + "\n", encoding="utf-8")

scfg = copy.deepcopy(cfg)                      # same settings, sample-only folders
scfg.paths.sources_file = str(SAMPLE / "sources.txt")
scfg.paths.input_audios = str(SAMPLE / "input_audios")
scfg.paths.output       = str(SAMPLE / "output")
scfg.paths.state_dir    = str(SAMPLE / ".state")
scfg.download.sleep_min_s = scfg.download.sleep_max_s = 0   # one video: no pacing needed
SAMPLE_OK = False
print("sample workspace:", SAMPLE)

### A1 · Download

In [ ]:
run_download(scfg)

audios = sorted(resolve(scfg.paths.input_audios).glob("*.*"))
assert audios, "❌ nothing downloaded; see the messages above"
a = audios[0]
sr, ch, dur = probe(a)
print(f"\n✅ {a.name}: {dur/60:.1f} min · {sr} Hz · {ch} ch")
import librosa
y, _ = librosa.load(a, sr=None, offset=min(60, dur / 2), duration=15)
display(Audio(y, rate=sr))                     # 15 s preview

### A2 · Diarization + chunking

In [ ]:
diarize_stage.run(scfg)
free_gpu()

import csv
out = resolve(scfg.paths.output)
rows = [r for man in out.glob("*/manifest.csv") for r in csv.DictReader(open(man, encoding="utf-8"))]
durs = [float(r["end"]) - float(r["start"]) for r in rows]
speakers = sorted({r["speaker"] for r in rows})
print(f"\n{len(rows)} chunks · {sum(durs)/60:.1f} min of speech · speakers: {speakers}")
if durs:
    print(f"chunk length: min {min(durs):.1f}s · mean {sum(durs)/len(durs):.1f}s · max {max(durs):.1f}s")

### A3 · Loudness normalization

In [ ]:
loudness.run(scfg)

### A4 · Quality scoring + filtering

In [ ]:
quality.run(scfg)
free_gpu()

out, all_recs, clean_recs = load_records(scfg)
kept = {r["chunk_path"] for r in clean_recs}
print(f"\nkept {len(clean_recs)}/{len(all_recs)} chunks "
      f"({hours(clean_recs)*60:.1f} of {hours(all_recs)*60:.1f} min) with "
      f"{scfg.quality.filter_metric} >= {scfg.quality.filter_threshold}")
try:
    import pandas as pd
    df = pd.DataFrame([{"chunk": r["chunk_path"].split("/")[-1], "dur": r["duration"],
                        **{k: round(v, 2) for k, v in r["scores"].items() if isinstance(v, (int, float))},
                        "kept": r["chunk_path"] in kept} for r in all_recs])
    display(df.sort_values(scfg.quality.filter_metric if scfg.quality.filter_metric in df else "dur"))
except ImportError:
    pass

### A5 · Transcription

In [ ]:
transcribe.run(scfg)
free_gpu()

### A6 · Check the sample

Automatic checks first. Then **listen**: kept chunks should be one clean voice, with no music,
no second speaker and no clipped words. Rejected chunks should really be worse. If the filter looks
too strict or too loose, change `quality.filter_threshold` in `config.yaml` and re-run from **Setup**.

In [ ]:
out, all_recs, clean_recs = load_records(scfg)
d = scfg.diarization
tol = 1.0                                       # padding can push edges slightly past the limits
sr, ch, _ = probe(sorted(resolve(scfg.paths.input_audios).glob("*.*"))[0])

checks = {
    f"audio is {scfg.download.sample_rate} Hz / {scfg.download.channels} ch": (sr, ch) == (scfg.download.sample_rate, scfg.download.channels),
    "diarization produced chunks": len(all_recs) > 0,
    f"all chunks {d.min_chunk_s}–{d.max_chunk_s}s (±{tol}s)":
        all(d.min_chunk_s - tol <= r["duration"] <= d.max_chunk_s + tol for r in all_recs),
    "every chunk has quality scores": all(r["scores"] for r in all_recs),
    "some chunks passed the quality filter": len(clean_recs) > 0,
    f"every kept chunk has {scfg.quality.filter_metric} >= {scfg.quality.filter_threshold}":
        all((metric(scfg, r) or 0) >= scfg.quality.filter_threshold for r in clean_recs),
    "every kept chunk is transcribed": all((r.get("text") or "").strip() for r in clean_recs),
}
for name, good in checks.items():
    print(("✅" if good else "❌"), name)
SAMPLE_OK = all(checks.values())
print("\nSAMPLE_OK =", SAMPLE_OK)

random.seed(0)
listen(scfg, clean_recs, 5, "🎧 KEPT chunks (random)")
kept = {r["chunk_path"] for r in clean_recs}
rejected = [r for r in all_recs if r["chunk_path"] not in kept]
listen(scfg, rejected, 3, "🗑️ REJECTED chunks (lowest score)", sort_low=True)

---
# Part B · Full run (all of `sources.txt`)

Only start this after Part A passed and the chunks sounded right. Each cell is one stage.
The download paces itself (random 3–8 s between videos) so the IP isn't flagged. With ~1000 videos it takes hours.
If a cell stops (a disconnect, a YouTube block), just run it again: finished work is skipped.

In [ ]:
FORCE = False          # set True only to run Part B without a passing sample test
assert SAMPLE_OK or FORCE, "Run Part A first and make sure it ends with SAMPLE_OK = True."
print("✅ go")

### B1 · Download all videos

In [ ]:
run_download(cfg)
audios = sorted(resolve(cfg.paths.input_audios).glob("*.*"))
print(f"\n{len(audios)} audio file(s) in {resolve(cfg.paths.input_audios)}")

### B2 · Diarization + chunking

In [ ]:
diarize_stage.run(cfg)
free_gpu()

### B3 · Loudness normalization

In [ ]:
loudness.run(cfg)

### B4 · Quality scoring + filtering

In [ ]:
quality.run(cfg)
free_gpu()

### B5 · Transcription

In [ ]:
transcribe.run(cfg)
free_gpu()

### B6 · Summary + spot check

In [ ]:
out, all_recs, clean_recs = load_records(cfg)
texted = [r for r in clean_recs if (r.get("text") or "").strip()]
videos = {r["chunk_path"].split("/")[0] for r in clean_recs}
print(f"all chunks  : {len(all_recs):>7}  · {hours(all_recs):7.2f} h")
print(f"clean chunks: {len(clean_recs):>7}  · {hours(clean_recs):7.2f} h  from {len(videos)} videos")
print(f"transcribed : {len(texted):>7}  · {hours(texted):7.2f} h  ← this is what gets published")
random.seed()
listen(cfg, texted, 5, "🎧 random clean chunks")

### B7 · Publish to HuggingFace (private)

Uploads the transcribed clean chunks to `publish.repo_id` in `config.yaml` as a **private** dataset.
Log in once first, in the RunPod terminal: `hf auth login` (the token needs **Write** access).
The upload is resumable, so if it stops, run this cell again.

In [ ]:
publish.run(cfg)